# Spike — validação do NotificadorTabela

Confirma que NotificadorTabela grava corretamente em observability.alertas, incluindo criação automática da tabela na primeira chamada.

Referências: ADR-003 (OOP), ADR-007 (alertas).

In [0]:
from src.observabilidade.notificadores import NotificadorTabela

notificador = NotificadorTabela(spark=spark)

alerta_teste = {
    "tipo_evento": "cadeia_fria",
    "origem": "observability_cadeia_fria",
    "severidade": "alta",
    "mensagem": "Teste — veículo incorreto detectado",
    "detalhes": {"remessa_id": "REM-TESTE-0001", "tipo_violacao": "veiculo_incorreto"},
}

resultado = notificador.notificar(alerta_teste)
print("Sucesso:", resultado)

spark.table("poc_pulse_observability.observability.alertas").show(truncate=False)

In [0]:
df_violacoes = spark.table("poc_pulse_observability.observability.observability_cadeia_fria").filter(
    "tipo_violacao = 'veiculo_incorreto'"
)

total_violacoes = df_violacoes.count()
print(f"Violações reais de veículo incorreto: {total_violacoes}")

if total_violacoes > 0:
    alerta_real = {
        "tipo_evento": "cadeia_fria",
        "origem": "observability_cadeia_fria",
        "severidade": "alta",
        "mensagem": f"{total_violacoes} remessas com veículo incorreto para produto de cadeia fria",
        "detalhes": {"total_violacoes": total_violacoes},
    }
    notificador.notificar(alerta_real)
    print("Alerta real registrado.")